# Difference-in-Differences

## Overview

Difference-in-Differences (DiD) estimates causal effects using panel data by comparing the change in outcomes over time for treated vs untreated units. It eliminates time-invariant confounding by differencing away fixed effects.

**The estimator:**
```
DiD = (Ȳ_treated_post - Ȳ_treated_pre) - (Ȳ_control_post - Ȳ_control_pre)
```

**Key identifying assumption — Parallel Trends:**  
In the absence of treatment, treated and control groups would have followed the same trajectory over time. This is untestable for the post-treatment period but can be assessed using pre-treatment data.

**When DiD applies:**
- A policy or intervention was applied to some units at a known time
- Control units were unaffected by the intervention
- Pre-treatment data exists for both groups

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

rng = np.random.default_rng(42)
n_sites  = 80
n_periods = 8   # 4 pre, 4 post
# Sites 0-39 receive intervention at period 5
treated_sites = np.arange(40)
control_sites = np.arange(40, 80)
true_DiD = 4.0   # true treatment effect

rows = []
for site in range(n_sites):
    is_treated = int(site < 40)
    site_fe = rng.normal(0, 3)          # time-invariant site characteristic
    for t in range(1, n_periods+1):
        post     = int(t >= 5)
        trend    = 0.5*t                 # common time trend
        treat_eff = true_DiD if (is_treated and post) else 0
        y = 18 + site_fe + trend + treat_eff + rng.normal(0, 1.5)
        rows.append({"site":site, "period":t, "treated":is_treated,
                      "post":post, "richness":y})
df = pd.DataFrame(rows)
print(f"Panel: {n_sites} sites x {n_periods} periods = {len(df)} observations")
print(f"True DiD effect: {true_DiD}")

---
## Visual Inspection and Parallel Trends Check

In [ ]:
means = df.groupby(["period","treated"])["richness"].mean().reset_index()
fig, ax = plt.subplots(figsize=(10,5))
for t, color, label in [(1,"#e74c3c","Treated"),(0,"steelblue","Control")]:
    sub = means[means["treated"]==t]
    ax.plot(sub["period"], sub["richness"], "o-", color=color, lw=2, label=label)
ax.axvline(4.5, color="grey", lw=1.5, linestyle="--", label="Intervention (period 5)")
ax.set_xlabel("Period"); ax.set_ylabel("Mean richness")
ax.set_title("DiD: Parallel pre-trends, divergence post-intervention")
ax.legend(); plt.tight_layout(); plt.show()
# Test parallel pre-trends: regression on pre-treatment data
pre_df = df[df["post"]==0].copy()
trend_model = smf.ols("richness ~ period * C(treated)", pre_df).fit()
interact_coef = trend_model.params.get("period:C(treated)[T.1]", None)
interact_p    = trend_model.pvalues.get("period:C(treated)[T.1]", None)
print(f"Pre-trend interaction (period x treated): coef={interact_coef:.3f}, p={interact_p:.4f}")
print("p > 0.05 -> no evidence of differential pre-trends (parallel trends plausible)")

---
## DiD Regression Estimator

In [ ]:
# Canonical DiD: Y = alpha + beta1*Treated + beta2*Post + beta3*(Treated x Post) + e
model_basic = smf.ols("richness ~ treated + post + treated:post", df).fit()
did_est = model_basic.params["treated:post"]
did_ci  = model_basic.conf_int().loc["treated:post"]
print("Basic DiD regression:")
print(f"  DiD estimate: {did_est:.3f}  (true={true_DiD})")
print(f"  95% CI:       [{did_ci[0]:.3f}, {did_ci[1]:.3f}]")
print(f"  p-value:      {model_basic.pvalues['treated:post']:.4f}")
# Two-Way Fixed Effects (TWFE): adds site and period FEs -- more efficient
model_twfe = smf.ols("richness ~ treated:post + C(site) + C(period)", df).fit()
did_twfe = model_twfe.params["treated:post"]
did_twfe_ci = model_twfe.conf_int().loc["treated:post"]
print(f"\nTWFE DiD estimate: {did_twfe:.3f}  (true={true_DiD})")
print(f"  95% CI: [{did_twfe_ci[0]:.3f}, {did_twfe_ci[1]:.3f}]")
print("TWFE absorbs site-level and period-level variation -> more precise estimate")

---
## Event Study Plot

In [ ]:
# Event study: estimate DiD coefficient for each period relative to pre-treatment
# Reference period = 4 (last pre-treatment period)
event_coefs, event_cis = [], []
df["period_rel"] = df["period"] - 4   # relative to period 4
for t in sorted(df["period_rel"].unique()):
    if t == 0:
        event_coefs.append(0); event_cis.append((0,0)); continue
    sub = df[df["period_rel"].isin([0,t])].copy()
    sub["post_t"] = (sub["period_rel"]==t).astype(int)
    m = smf.ols("richness ~ treated*post_t + C(site) + C(period)", sub).fit()
    key = "treated:post_t"
    coef = m.params.get(key, np.nan)
    ci   = m.conf_int().loc[key] if key in m.conf_int().index else (np.nan,np.nan)
    event_coefs.append(coef); event_cis.append(ci)
periods_rel = sorted(df["period_rel"].unique())
fig, ax = plt.subplots(figsize=(10,5))
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(-0.5, color="grey", lw=1.5, linestyle="--", label="Intervention")
ax.axhline(true_DiD, color="navy", lw=1.5, linestyle=":", label=f"True effect={true_DiD}")
for t, coef, (lo,hi) in zip(periods_rel, event_coefs, event_cis):
    color = "#e74c3c" if t > 0 else "steelblue"
    ax.scatter(t, coef, s=60, color=color, zorder=4)
    if not np.isnan(lo):
        ax.plot([t,t],[lo,hi], color=color, lw=2)
ax.set_xlabel("Period relative to intervention"); ax.set_ylabel("DiD coefficient")
ax.set_title("Event Study: Pre-trends flat, post-treatment effect emerges")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Clustered standard errors: sites within a catchment share errors
# Standard errors must be clustered at the treatment assignment level
model_clustered = smf.ols("richness ~ treated + post + treated:post", df).fit(
    cov_type="cluster", cov_kwds={"groups": df["site"]})
did_cl = model_clustered.params["treated:post"]
did_cl_se = model_clustered.bse["treated:post"]
did_cl_ci = model_clustered.conf_int().loc["treated:post"]
print("Clustered SE DiD (clustered by site):")
print(f"  DiD estimate: {did_cl:.3f}, SE={did_cl_se:.3f}")
print(f"  95% CI: [{did_cl_ci[0]:.3f}, {did_cl_ci[1]:.3f}]")
print("\nAlways cluster SEs at the level of treatment assignment")
print("Heteroscedasticity-only robust SEs are insufficient for panel DiD")

---

## Common Pitfalls

**1. Not checking parallel pre-trends**  
Parallel trends is the core identifying assumption of DiD. Always plot group means over all pre-treatment periods and run a formal interaction test. If groups were already diverging before the intervention, DiD does not identify the causal effect.

**2. Ignoring staggered treatment timing**  
When different units receive treatment at different times (staggered DiD), the standard TWFE estimator is biased — it uses early-treated units as controls for late-treated units. Use Callaway-Sant'Anna or Sun-Abraham estimators for staggered designs.

**3. Not clustering standard errors at the treatment level**  
When treatment is assigned at the group level (e.g. catchment or policy region), observations within the same group share correlated errors. Clustering SEs at the treatment assignment level is necessary; heteroscedasticity-robust SEs alone are insufficient.

**4. Interpreting a significant DiD as ruling out all confounders**  
DiD eliminates time-invariant confounders but not time-varying ones. If a concurrent event affected treated and control groups differently at the same time as the intervention, DiD conflates the treatment effect with the concurrent event.

**5. Applying DiD to a single treated unit**  
With only one treated unit and multiple controls, the estimate is identified but inference is invalid — standard errors cannot be computed reliably. Use permutation inference (randomisation test) or synthetic control instead.
---
*python_methods_library - Samantha McGarrigle*